This notebook shows two ways to solve a linear programming problem using Highspy. The first way is to use the Highspy library directly, and the second way is to use the Highs solver from the Pyomo library.

Here is the mathematical formulation of the LP problem:

Minimize    z = x₁ + x₂

Subject to:
            x₂ ≤ 7
            5 ≤ x₁ + 2x₂ ≤ 15
            3x₁ + 2x₂ ≥ 6
            
            0 ≤ x₁ ≤ 4
            x₂ ≥ 1

Optimal solution:
    objective = 2.75
    x₁ = 0.5
    x₂ = 2.25



## Highspy

In [2]:
import highspy
import numpy as np

# Highs h
h = highspy.Highs()

In [2]:
inf = highspy.kHighsInf
# Define a HighsLp instance
lp = highspy.HighsLp()
lp.num_col_ = 2;
lp.num_row_ = 3;
lp.col_cost_ = np.array([1, 1], dtype=np.double)
lp.col_lower_ = np.array([0, 1], dtype=np.double)
lp.col_upper_ = np.array([4, inf], dtype=np.double)
lp.row_lower_ = np.array([-inf, 5, 6], dtype=np.double)
lp.row_upper_ = np.array([7, 15, inf], dtype=np.double)
# In a HighsLp instsance, the number of nonzeros is given by a fictitious final start
lp.a_matrix_.start_ = np.array([0, 2, 5])
lp.a_matrix_.index_ = np.array([1, 2, 0, 1, 2])
lp.a_matrix_.value_ = np.array([1, 3, 1, 2, 2], dtype=np.double)
h.passModel(lp)

Running HiGHS 1.8.1 (git hash: 4a7f24a): Copyright (c) 2024 HiGHS under MIT licence terms


<HighsStatus.kOk: 0>

In [3]:

h.version()


output_flag = h.getOptionValue("output_flag")
print("output_flag:", output_flag)

output_flag = h.getOptionValue("log_to_console")
print("log_to_console:", output_flag)

output_flag = h.getOptionValue("log_dev_level")
print("log_dev_level:", output_flag)


h.setOptionValue('output_flag',False)

h.setOptionValue('log_dev_level',1)

h.setOptionValue('output_flag',True)

output_flag: (<HighsStatus.kOk: 0>, True)
log_to_console: (<HighsStatus.kOk: 0>, True)
log_dev_level: (<HighsStatus.kOk: 0>, 0)


<HighsStatus.kOk: 0>

In [4]:
h.run()

Coefficient ranges:
  Matrix [1e+00, 3e+00]
  Cost   [1e+00, 1e+00]
  Bound  [1e+00, 4e+00]
  RHS    [5e+00, 2e+01]
Presolving model
2 rows, 2 cols, 4 nonzeros  0s
2 rows, 2 cols, 4 nonzeros  0s
Presolve : Reductions: rows 2(-1); columns 2(-0); elements 4(-1)
Solving the presolved LP
Using EKK dual simplex solver - serial
Cost perturbation for 
   Initially have 2 nonzero costs (100%) with min / average / max = 1 / 1 / 1
   Perturbation column base = 5e-07
   Perturbation row    base = 1e-12
       Iteration        Objective     Infeasibilities num(sum)
DuPh2          0     1.0000013886e+00 Pr: 2(7) No reason
DuPh2          2     2.7500039668e+00 Pr: 0(0) Possibly optimal
DuPh2          2     2.7500000000e+00 Pr: 0(0) Perturbation cleanup
Simplex iterations: DuPh2 2; Total 2
EKK dual simplex solver returns 0 primal and 0 dual infeasibilities: Status Optimal
Solving the original LP from the solution after postsolve
Postsolve  : 0
Time       :     0.00
Time Pre   :     0.00
Time PreLP : 

<HighsStatus.kOk: 0>

In [5]:
solution = h.getSolution()
basis = h.getBasis()
info = h.getInfo()
model_status = h.getModelStatus()
print('Model status = ', h.modelStatusToString(model_status))
print()
print('Optimal objective = ', info.objective_function_value)
print('Iteration count = ', info.simplex_iteration_count)
print('Primal solution status = ', h.solutionStatusToString(info.primal_solution_status))
print('Dual solution status = ', h.solutionStatusToString(info.dual_solution_status))
print('Basis validity = ', h.basisValidityToString(info.basis_validity))

Model status =  Optimal

Optimal objective =  2.75
Iteration count =  2
Primal solution status =  Feasible
Dual solution status =  Feasible
Basis validity =  Valid


## Pyomo

In [35]:
import pyomo.environ as pyo

# Create a concrete model
model = pyo.ConcreteModel()

# Define variables with bounds
model.x1 = pyo.Var(bounds=(0, 4))  # 0 ≤ x₁ ≤ 4
model.x2 = pyo.Var(bounds=(1, None))  # x₂ ≥ 1

# Define the objective function
model.obj = pyo.Objective(expr=model.x1 + model.x2, sense=pyo.minimize)

# Define the constraints
model.con1 = pyo.Constraint(expr=model.x2 <= 7)  # x₂ ≤ 7
model.con2 = pyo.Constraint(expr=5 <= model.x1 + 2 * model.x2)  # 5 ≤ x₁ + 2x₂
model.con3 = pyo.Constraint(expr=model.x1 + 2 * model.x2 <= 15)  # x₁ + 2x₂ ≤ 15
model.con4 = pyo.Constraint(expr=3 * model.x1 + 2 * model.x2 >= 6)  # 3x₁ + 2x₂ ≥ 6

# Solve the problem using the HiGHS solver
# solver = pyo.SolverFactory('highs')
from pyomo.contrib.appsi.solvers import Highs

# # Create a solver instance
solver = Highs()
result = solver.solve(model)

# Display the results
if result.termination_condition.name == 'optimal':
    print("Optimal solution found:")
    print(f"x = {pyo.value(model.x1)}")
    print(f"y = {pyo.value(model.x2)}")
    # Print dual values for each constraint
    print("\nDual values:")
    print(f"con1: {next(iter(solver.get_duals([model.con1]).values()))}")
    print(f"con2: {next(iter(solver.get_duals([model.con2]).values()))}")
    print(f"con3: {next(iter(solver.get_duals([model.con3]).values()))}")
    print(f"con4: {next(iter(solver.get_duals([model.con4]).values()))}")
else:
    print("Solver did not find an optimal solution.")


Optimal solution found:
x = 0.5
y = 2.25

Dual values:
con1: -0.0
con2: 0.25
con3: -0.0
con4: 0.25
